# Checking Glider Formatting
This notebook provides a basic overview of how to use the `Format Checker` within `format_check.py`.

This step exists to check on the format of the file and all of the data therein for functionality and adherance to different *standards* as outlined in the [IOOS Compliance Checker tool](https://github.com/ioos/compliance-checker).

Relevant to any kind of ocean glider formatting is the [OG1 format](https://oceangliderscommunity.github.io/OG-format-user-manual/OG_Format.html), describing formatting for file attributes, units, and much more. OG1 does not come bundled in the IOOS Compliance Checker tool, so the [OG1 plugin](https://github.com/ioos/cc-plugin-og) must be installed.

Let's start by initializing the pipeline and downloading our data.

In [ ]:
from pathlib import Path
import xarray as xr
import os
import json

cwd = os.getcwd()
os.chdir(f"{cwd}/../../src")

input_dir = Path("../examples/data/OG1")
input_file = input_dir / "Nelson_646_R.nc"
if not input_dir.exists():
    input_dir.mkdir(parents=True)
if not input_file.exists():
    import requests
    response = requests.get("https://linkedsystems.uk/erddap/files/Public_OG1_Data_001/Nelson_20240528/Nelson_646_R.nc")
    if response.status_code == 200:
        with open(input_file, "wb") as f:
            f.write(response.content)
        print(f"Example file downloaded and written to {input_dir.resolve()}")
    else:
        print("File download failed")
dataset = xr.load_dataset("../examples/data/OG1/Nelson_646_R.nc")
dataset

In [ ]:
from pelagos_py.pipeline import *
Pipe = Pipeline(config_path = r"../examples/configs/example_config_formatchk.yaml")

In [ ]:
for i, step in enumerate(Pipe.steps):
    print(f"{i} {step["name"]}, \t {list(step["parameters"].keys())}")
    print(f"\t\t {list(step["parameters"].items())}")

As you can see in this pipeline, the first step that is run is the Format Checker. This is a good spot for it, since it can tell us if our dataset is compliant with whatever specified standards we have.

In [ ]:
print(Pipe.steps[0])

Unlike other pipeline steps, the checker can run independently and does not depend on any other steps. However, it's best to check the data for proper formatting prior to working with it and to still return `context` for consistency with other steps in the pipeline.

In [ ]:
context = Pipe.execute_step(Pipe.steps[0], None)

Now, the format checker is warning us that the file did not pass the file checker. That's not the end of the world - more so that there are some standards or recommendations that are not yet being followed. Since we explicitly told processing to continue, the pipeline didn't error out (example later).

At this point, we can see that a `Nelson_646_R_check.rst" file has been created in our testing folder.

This outlines what problems there are with the file, which may/may not be required for processing, hence the warning from the checker step.

In [ ]:
fname = "../examples/data/OG1/testing/Nelson_646_R_check.rst"
with open(fname, mode="r") as f:
    content = f.read()
    print(content)

If doing reports, this file will get appended to the final .rst and builds the same way as the rest of the document (`write_report.add_cc`).

Now we can run the remaining steps (1 to the end) and check on our report demo.

In [ ]:
for i in range(1,5):
    context = Pipe.execute_step(Pipe.steps[i], _context = context)

We can open our finalized report in the defined `demo.rst` and confirm that the checker file has been inserted.

In [ ]:
fname = "../examples/data/OG1/testing/formatchk_demo.rst"
with open(fname, mode="r") as f:
    content = f.read()
    content = content[-5855:]
    print(content)

But there is another way to export the format checker's results, as a JSON. This is a more-detailed way to export the results, with additional support for things like a performance score that can be extracted.

In [ ]:
Pipe.steps[0]['parameters']['output_type'] = 'json'
Pipe.steps[0]['parameters']['proceed_on_fail'] = True
print(Pipe.steps[0])

In [ ]:
Pipe.run()

Now if we were to load our format check file, it will be as a JSON file.

In [ ]:
with open("../examples/data/OG1/testing/Nelson_646_R_check.json") as f:
    cc_data = json.load(f)
print(cc_data)

This JSON data gets handled as a table, rather than ASCII line-by-line in the final reporting. Since we've reran it, our report `formatchk_demo.rst` should be different.

In [ ]:
fname = "../examples/data/OG1/testing/formatchk_demo.rst"
with open(fname, mode="r") as f:
    content = f.read()
    content = content[-5970:]
    print(content)

## Halting the pipeline

Now, let's assume there is an instance where processing should *not* proceed if the data are not entirely in the desired format. For this, we need only adjust `proceed_on_fail` in this instance and we should get a controlled error out.

In [ ]:
Pipe.steps[0]["parameters"]["proceed_on_fail"] = False
print(Pipe.steps[0]["parameters"])

In [ ]:
try:
    Pipe.execute_step(Pipe.steps[0], None)
except RuntimeError as e:
    print("\nAs you can see, we have the error in the log as well.")
    print("(a warning that it did not pass, and a log entry that the pipeline is being halted).")
    print(f"This differs from the RuntimeError: '{e}'")

This is handy if there are instances where you want to guarantee that your data is in the specified format. The JSON output from the format checker step can be used to identify particular error messages or scan for variables.

This makes it easier to establish controlled stops to the pipeline if the data isn't ready.